# Models

Second notebook in the split workflow. It loads the data artifacts from `01_data_pipeline.ipynb`, trains/evaluates the TF-IDF + Logistic Regression baselines and BiLSTM model, and writes model outputs under `baseline/` and `bilstm/`.


## 1. Configuration

In [1]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Shared data folder.
# -----------------------------------------------------------------------------
# All files that are inputs to, outputs from, or shared between multiple models
# are stored in data/. Model-specific artifacts remain in their own folders.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Input XML.
XML_PATH = DATA_DIR / "bookworm_09062026.xml"

# Shared preprocessing and training files used by the split workflow.
PAGES_CSV = DATA_DIR / "pages.csv"
CHARACTER_GAZETTEER_CSV = DATA_DIR / "character_gazetteer.csv"
CANDIDATE_EXAMPLES_CSV = DATA_DIR / "candidate_examples.csv"
LLM_LABELED_CANDIDATES_CSV = DATA_DIR / "candidate_examples_llm_labeled.csv"
WEAK_LABEL_DISTRIBUTION_CSV = DATA_DIR / "label_distribution.csv"
TRAINING_LABEL_DISTRIBUTION_CSV = DATA_DIR / "training_label_distribution.csv"
TRAIN_CSV = DATA_DIR / "train.csv"
DEV_CSV = DATA_DIR / "dev.csv"
TEST_CSV = DATA_DIR / "test.csv"
SPLIT_LABEL_DISTRIBUTION_CSV = DATA_DIR / "split_label_distribution.csv"
MODEL_COMPARISON_CSV = DATA_DIR / "metrics_model_comparison.csv"
KG_MODEL_COMPARISON_CSV = DATA_DIR / "kg_model_comparison.csv"
RUN_SUMMARY_JSON = DATA_DIR / "model_run_summary.json"
PIPELINE_RUN_METADATA_JSON = DATA_DIR / "pipeline_run_metadata.json"

# Model-specific output folders.
BASELINE_DIR = Path("baseline")  # TF-IDF + Logistic Regression artifacts.
BILSTM_DIR = Path("bilstm")      # BiLSTM artifacts.
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BILSTM_DIR.mkdir(parents=True, exist_ok=True)

# Relationship labels. "no_relation" is needed as the negative class.
RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]

NO_RELATION_LABEL = "no_relation"
POSITIVE_RELATIONS = [label for label in RELATIONSHIPS if label != NO_RELATION_LABEL]

RANDOM_SEED = 42
TEST_SIZE = 0.15
DEV_SIZE = 0.15
MAX_NEGATIVE_RATIO = 2.0
MIN_CONTEXT_CHARS = 25
EDGE_CONFIDENCE_THRESHOLD = 0.95  # predictions_all_best_baseline.csv

print(f"XML path: {XML_PATH.resolve()}")
print(f"Shared data folder: {DATA_DIR.resolve()}")
print(f"Baseline output folder: {BASELINE_DIR.resolve()}")
print(f"BiLSTM output folder: {BILSTM_DIR.resolve()}")
print(f"Relationship labels: {RELATIONSHIPS}")


XML path: E:\Natural Language Processing\Project 2\data\bookworm_09062026.xml
Shared data folder: E:\Natural Language Processing\Project 2\data
Baseline output folder: E:\Natural Language Processing\Project 2\baseline
BiLSTM output folder: E:\Natural Language Processing\Project 2\bilstm
Relationship labels: ['family', 'romantic', 'friend_ally', 'service_retainer', 'enemy_rival', 'no_relation']


## 2. Imports

In [2]:
import html
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
np.random.seed(RANDOM_SEED)

## Load artifacts from 01_data_pipeline.ipynb

Run the data-pipeline notebook first. This cell restores the shared candidate and split DataFrames that the model-training cells expect.


In [3]:
def require_file(path: Path, upstream_notebook: str) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run {upstream_notebook} first.")
    return path

pipeline_metadata = json.loads(PIPELINE_RUN_METADATA_JSON.read_text(encoding="utf-8")) if PIPELINE_RUN_METADATA_JSON.exists() else {}

candidate_source_csv = Path(pipeline_metadata.get("candidate_source_csv", CANDIDATE_EXAMPLES_CSV))
LABEL_SOURCE = pipeline_metadata.get("label_source", "weak_labels")

candidates_df = pd.read_csv(require_file(candidate_source_csv, "01_data_pipeline.ipynb"))
train_df = pd.read_csv(require_file(TRAIN_CSV, "01_data_pipeline.ipynb"))
dev_df = pd.read_csv(require_file(DEV_CSV, "01_data_pipeline.ipynb"))
test_df = pd.read_csv(require_file(TEST_CSV, "01_data_pipeline.ipynb"))
split_method = pipeline_metadata.get("split_method", "loaded_from_csv")

required_columns = {"candidate_id", "head", "tail", "context", "label", "weak_label", "source_type", "pair_id", "text_basic", "text_marked"}
for frame_name, frame in {
    "candidates_df": candidates_df,
    "train_df": train_df,
    "dev_df": dev_df,
    "test_df": test_df,
}.items():
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise ValueError(f"{frame_name} is missing required columns: {missing}")

print(f"Loaded candidates: {len(candidates_df):,} rows ({LABEL_SOURCE}) from {candidate_source_csv}")
print(f"Loaded split: train={len(train_df):,}, dev={len(dev_df):,}, test={len(test_df):,}; split_method={split_method}")


Loaded candidates: 6,031 rows (llm_judge) from data\candidate_examples_llm_labeled.csv
Loaded split: train=4,251, dev=915, test=865; split_method=grouped_by_pair


## 10. Train TF-IDF + Logistic Regression baseline variations

This notebook trains two variants of the same baseline model family:

1. `tfidf_logreg_basic`: TF-IDF over the raw context only.
2. `tfidf_logreg_marked`: TF-IDF over entity-marked context with section/source metadata.

The second version corresponds to the feature-engineering variation from the proposal.

In [4]:
def build_tfidf_logreg_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 3),
                    min_df=1,
                    max_df=0.95,
                    max_features=50_000,
                    sublinear_tf=True,
                    strip_accents="unicode",
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    C=10.0,
                    class_weight=None,
                    solver="lbfgs",
                    tol=1e-4,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )

def evaluate_predictions(y_true, y_pred, labels_order: list[str]) -> dict:
    labels_present = [label for label in labels_order if label in set(y_true) | set(y_pred)]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels_present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels_present, average="weighted", zero_division=0),
    }


def train_and_evaluate_variant(variant_name: str, text_column: str):
    print(f"\nTraining variant: {variant_name}")
    model = build_tfidf_logreg_pipeline()
    model.fit(train_df[text_column], train_df["label"])

    dev_pred = model.predict(dev_df[text_column])
    test_pred = model.predict(test_df[text_column])

    dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)
    test_metrics = evaluate_predictions(test_df["label"], test_pred, RELATIONSHIPS)

    metrics_row = {
        "variant": variant_name,
        "text_column": text_column,
        "dev_accuracy": dev_metrics["accuracy"],
        "dev_macro_f1": dev_metrics["macro_f1"],
        "dev_weighted_f1": dev_metrics["weighted_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "train_examples": len(train_df),
        "dev_examples": len(dev_df),
        "test_examples": len(test_df),
    }

    # Save model.
    model_path = BASELINE_DIR / f"{variant_name}.joblib"
    joblib.dump(model, model_path)

    # Save classification report and confusion matrix for the test set.
    labels_present = [label for label in RELATIONSHIPS if label in set(test_df["label"]) | set(test_pred)]
    report = classification_report(
        test_df["label"],
        test_pred,
        labels=labels_present,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(BASELINE_DIR / f"classification_report_{variant_name}.csv")

    cm = confusion_matrix(test_df["label"], test_pred, labels=labels_present)
    cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels_present], columns=[f"pred_{x}" for x in labels_present])
    cm_df.to_csv(BASELINE_DIR / f"confusion_matrix_{variant_name}.csv")

    # Save test predictions.
    test_out = test_df.copy()
    test_out["predicted_label"] = test_pred
    if hasattr(model.named_steps["clf"], "predict_proba"):
        probs = model.predict_proba(test_df[text_column])
        classes = model.named_steps["clf"].classes_
        test_out["confidence"] = probs.max(axis=1)
        for i, label in enumerate(classes):
            test_out[f"prob_{label}"] = probs[:, i]
    test_out.to_csv(BASELINE_DIR / f"predictions_test_{variant_name}.csv", index=False)

    return model, metrics_row


variants = {
    "tfidf_logreg_basic": "text_basic",
    "tfidf_logreg_marked": "text_marked",
}

trained_models = {}
metrics_rows = []
for variant_name, text_column in variants.items():
    model, metrics = train_and_evaluate_variant(variant_name, text_column)
    trained_models[variant_name] = {"model": model, "text_column": text_column}
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).sort_values("dev_macro_f1", ascending=False).reset_index(drop=True)
metrics_df.to_csv(BASELINE_DIR / "metrics_baseline_variants.csv", index=False)

display(metrics_df)


Training variant: tfidf_logreg_basic

Training variant: tfidf_logreg_marked


,variant,text_column,dev_accuracy,dev_macro_f1,dev_weighted_f1,test_accuracy,test_macro_f1,test_weighted_f1,train_examples,dev_examples,test_examples
0,tfidf_logreg_basic,text_basic,0.718033,0.678755,0.715505,0.694798,0.637746,0.689746,4251,915,865
1,tfidf_logreg_marked,text_marked,0.713661,0.668250,0.710262,0.702890,0.649758,0.696280,4251,915,865


## 11. Train BiLSTM relation classifier

This section implements the second model: a basic neural relation classifier using a bidirectional LSTM.

The model reuses the same `train_df`, `dev_df`, and `test_df` splits as the TF-IDF baseline. It uses the `text_marked` column, which includes explicit `[HEAD]` and `[TAIL]` markers so the model knows which two characters the label refers to.

Architecture:

```text
entity-marked text
  -> regex tokenizer
  -> vocabulary IDs
  -> embedding layer
  -> bidirectional LSTM
  -> final forward/backward hidden states
  -> dropout
  -> linear classification layer
  -> softmax relationship label
```


In [5]:
# BiLSTM relation classifier.
# This cell intentionally uses only the existing train/dev/test DataFrames so the comparison
# with the baseline is made on exactly the same split.

import copy
import random
from collections import Counter

try:
    import torch
    import torch.nn as nn
    from torch.nn.utils.rnn import pack_padded_sequence
    from torch.utils.data import DataLoader, Dataset
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for the BiLSTM section. Install torch in the notebook environment and rerun this cell."
    ) from exc

# BILSTM_DIR is configured in Section 1 so model-specific outputs stay separate from shared data files.
BILSTM_DIR.mkdir(parents=True, exist_ok=True)

BILSTM_TEXT_COLUMN = "text_marked"
BILSTM_MAX_VOCAB_SIZE = 20_000
BILSTM_MIN_TOKEN_FREQ = 1
BILSTM_MAX_LEN = 128
BILSTM_EMBEDDING_DIM = 100
BILSTM_HIDDEN_DIM = 128
BILSTM_NUM_LAYERS = 1
BILSTM_DROPOUT = 0.30
BILSTM_BATCH_SIZE = min(32, max(2, len(train_df)))
BILSTM_EPOCHS = 16
BILSTM_LEARNING_RATE = 1e-3
BILSTM_WEIGHT_DECAY = 1e-5
BILSTM_PATIENCE = 2

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Limiting CPU threads avoids occasional OpenMP/MKL slowdowns or deadlocks on small notebook workloads.
if DEVICE.type == "cpu":
    torch.set_num_threads(1)
print(f"BiLSTM device: {DEVICE}")


def set_reproducible_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_reproducible_seed(RANDOM_SEED)

# Keep entity markers as their own tokens. Lowercasing makes the vocabulary smaller.
TOKEN_PATTERN = re.compile(r"\[/?(?:HEAD|TAIL)\]|[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]", flags=re.IGNORECASE)


def tokenize_for_bilstm(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_PATTERN.findall(str(text or ""))]


def build_bilstm_vocab(texts: pd.Series, max_vocab_size: int = BILSTM_MAX_VOCAB_SIZE, min_freq: int = BILSTM_MIN_TOKEN_FREQ) -> dict[str, int]:
    counts = Counter()
    for text in texts.fillna(""):
        counts.update(tokenize_for_bilstm(text))

    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for token, count in counts.most_common(max_vocab_size - len(vocab)):
        if count >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab


vocab = build_bilstm_vocab(train_df[BILSTM_TEXT_COLUMN])
label_to_id = {label: idx for idx, label in enumerate(RELATIONSHIPS)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print(f"BiLSTM vocabulary size: {len(vocab):,}")
print(f"Number of labels: {len(label_to_id)}")


class RelationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_column: str, vocab: dict[str, int], label_to_id: dict[str, int], max_len: int):
        self.df = df.reset_index(drop=True).copy()
        self.text_column = text_column
        self.vocab = vocab
        self.label_to_id = label_to_id
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.df)

    def encode_text(self, text: str) -> list[int]:
        tokens = tokenize_for_bilstm(text)[: self.max_len]
        if not tokens:
            tokens = [UNK_TOKEN]
        return [self.vocab.get(token, UNK_IDX) for token in tokens]

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        input_ids = self.encode_text(row[self.text_column])
        label_id = self.label_to_id[row["label"]]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "length": len(input_ids),
            "label": torch.tensor(label_id, dtype=torch.long),
        }


def collate_relation_batch(batch: list[dict]) -> dict[str, torch.Tensor]:
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    labels = torch.stack([item["label"] for item in batch])
    max_len = int(lengths.max().item())

    input_ids = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)
    for row_idx, item in enumerate(batch):
        ids = item["input_ids"]
        input_ids[row_idx, : len(ids)] = ids

    return {"input_ids": input_ids, "lengths": lengths, "labels": labels}


def make_bilstm_loader(df: pd.DataFrame, shuffle: bool = False) -> DataLoader:
    dataset = RelationDataset(df, BILSTM_TEXT_COLUMN, vocab, label_to_id, BILSTM_MAX_LEN)
    return DataLoader(
        dataset,
        batch_size=BILSTM_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_relation_batch,
    )


class BiLSTMRelationClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        embedding_dim: int = BILSTM_EMBEDDING_DIM,
        hidden_dim: int = BILSTM_HIDDEN_DIM,
        num_layers: int = BILSTM_NUM_LAYERS,
        dropout: float = BILSTM_DROPOUT,
        pad_idx: int = PAD_IDX,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed)

        # hidden shape: [num_layers * 2, batch, hidden_dim]
        # Last layer forward state is hidden[-2], backward state is hidden[-1].
        representation = torch.cat([hidden[-2], hidden[-1]], dim=1)
        representation = self.dropout(representation)
        return self.classifier(representation)


def predict_bilstm(model: nn.Module, df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    loader = make_bilstm_loader(df, shuffle=False)
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            logits = model(input_ids, lengths)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            all_probs.append(probs)

    probs = np.vstack(all_probs) if all_probs else np.empty((0, len(RELATIONSHIPS)))
    pred_ids = probs.argmax(axis=1) if len(probs) else np.array([], dtype=int)
    pred_labels = np.array([id_to_label[int(idx)] for idx in pred_ids])
    return pred_labels, probs


train_loader = make_bilstm_loader(train_df, shuffle=True)

y_counts = train_df["label"].value_counts()
class_weights = []
for label in RELATIONSHIPS:
    # Use inverse-frequency weighting so minority relation classes matter during training.
    count = max(1, int(y_counts.get(label, 0)))
    class_weights.append(len(train_df) / (len(RELATIONSHIPS) * count))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

bilstm_model = BiLSTMRelationClassifier(
    vocab_size=len(vocab),
    num_labels=len(RELATIONSHIPS),
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    bilstm_model.parameters(),
    lr=BILSTM_LEARNING_RATE,
    weight_decay=BILSTM_WEIGHT_DECAY,
)

best_dev_macro_f1 = -1.0
best_state = None
epochs_without_improvement = 0
bilstm_history_rows = []

for epoch in range(1, BILSTM_EPOCHS + 1):
    bilstm_model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = bilstm_model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

    train_loss = total_loss / max(1, total_examples)
    dev_pred, _ = predict_bilstm(bilstm_model, dev_df)
    dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)

    bilstm_history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_accuracy": dev_metrics["accuracy"],
            "dev_macro_f1": dev_metrics["macro_f1"],
            "dev_weighted_f1": dev_metrics["weighted_f1"],
        }
    )

    print(
        f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
        f"dev_macro_f1={dev_metrics['macro_f1']:.4f} | dev_accuracy={dev_metrics['accuracy']:.4f}"
    )

    if dev_metrics["macro_f1"] > best_dev_macro_f1:
        best_dev_macro_f1 = dev_metrics["macro_f1"]
        best_state = copy.deepcopy(bilstm_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= BILSTM_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

if best_state is not None:
    bilstm_model.load_state_dict(best_state)

bilstm_history_df = pd.DataFrame(bilstm_history_rows)
bilstm_history_df.to_csv(BILSTM_DIR / "training_history.csv", index=False)

dev_pred, dev_probs = predict_bilstm(bilstm_model, dev_df)
test_pred, test_probs = predict_bilstm(bilstm_model, test_df)

dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)
test_metrics = evaluate_predictions(test_df["label"], test_pred, RELATIONSHIPS)

bilstm_metrics = {
    "variant": "bilstm_marked",
    "text_column": BILSTM_TEXT_COLUMN,
    "dev_accuracy": dev_metrics["accuracy"],
    "dev_macro_f1": dev_metrics["macro_f1"],
    "dev_weighted_f1": dev_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "train_examples": len(train_df),
    "dev_examples": len(dev_df),
    "test_examples": len(test_df),
    "vocab_size": len(vocab),
    "max_len": BILSTM_MAX_LEN,
    "embedding_dim": BILSTM_EMBEDDING_DIM,
    "hidden_dim": BILSTM_HIDDEN_DIM,
    "epochs_run": int(len(bilstm_history_df)),
    "best_dev_macro_f1": float(best_dev_macro_f1),
}

bilstm_metrics_df = pd.DataFrame([bilstm_metrics])
bilstm_metrics_df.to_csv(BILSTM_DIR / "metrics_bilstm.csv", index=False)

labels_present = [label for label in RELATIONSHIPS if label in set(test_df["label"]) | set(test_pred)]
bilstm_report = classification_report(
    test_df["label"],
    test_pred,
    labels=labels_present,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(bilstm_report).transpose().to_csv(BILSTM_DIR / "classification_report_bilstm.csv")

bilstm_cm = confusion_matrix(test_df["label"], test_pred, labels=labels_present)
bilstm_cm_df = pd.DataFrame(
    bilstm_cm,
    index=[f"true_{label}" for label in labels_present],
    columns=[f"pred_{label}" for label in labels_present],
)
bilstm_cm_df.to_csv(BILSTM_DIR / "confusion_matrix_bilstm.csv")

bilstm_test_out = test_df.copy()
bilstm_test_out["predicted_label"] = test_pred
bilstm_test_out["confidence"] = test_probs.max(axis=1) if len(test_probs) else []
for label_idx, label in enumerate(RELATIONSHIPS):
    bilstm_test_out[f"prob_{label}"] = test_probs[:, label_idx] if len(test_probs) else []
bilstm_test_out.to_csv(BILSTM_DIR / "predictions_test_bilstm.csv", index=False)

# Save a checkpoint and the vocabulary so the model can be reused later.
torch.save(
    {
        "model_state_dict": bilstm_model.state_dict(),
        "vocab": vocab,
        "label_to_id": label_to_id,
        "config": {
            "max_len": BILSTM_MAX_LEN,
            "embedding_dim": BILSTM_EMBEDDING_DIM,
            "hidden_dim": BILSTM_HIDDEN_DIM,
            "num_layers": BILSTM_NUM_LAYERS,
            "dropout": BILSTM_DROPOUT,
        },
    },
    BILSTM_DIR / "bilstm_relation_classifier.pt",
)

# Predict all candidates so the same KG aggregation code can be reused later.
bilstm_all_pred = candidates_df.copy()
bilstm_all_labels, bilstm_all_probs = predict_bilstm(bilstm_model, bilstm_all_pred)
bilstm_all_pred["predicted_label"] = bilstm_all_labels
bilstm_all_pred["confidence"] = bilstm_all_probs.max(axis=1) if len(bilstm_all_probs) else []
for label_idx, label in enumerate(RELATIONSHIPS):
    bilstm_all_pred[f"prob_{label}"] = bilstm_all_probs[:, label_idx] if len(bilstm_all_probs) else []

# Keep structured infobox relations as high-confidence weak labels only during weak-label runs.
# When LLM Judge labels are enabled, avoid reintroducing weak labels into KG generation.
if LABEL_SOURCE == "weak_labels":
    structured_mask = (
        (bilstm_all_pred["source_type"] == "infobox")
        & (bilstm_all_pred["weak_label"] != NO_RELATION_LABEL)
    )
    bilstm_all_pred.loc[structured_mask, "predicted_label"] = bilstm_all_pred.loc[structured_mask, "weak_label"]
    bilstm_all_pred.loc[structured_mask, "confidence"] = 0.95
else:
    print("LLM Judge labels are enabled; skipping infobox weak-label override for BiLSTM KG predictions.")

bilstm_all_predictions_path = BILSTM_DIR / "predictions_all_bilstm.csv"
bilstm_all_pred.to_csv(bilstm_all_predictions_path, index=False)

print(f"Saved BiLSTM outputs to: {BILSTM_DIR}")
display(bilstm_metrics_df)
display(bilstm_history_df.tail())
display(bilstm_test_out[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))


BiLSTM device: cpu
BiLSTM vocabulary size: 6,378
Number of labels: 6
Epoch 01 | loss=1.6256 | dev_macro_f1=0.3837 | dev_accuracy=0.4536
Epoch 02 | loss=1.3511 | dev_macro_f1=0.4674 | dev_accuracy=0.5224
Epoch 03 | loss=1.0927 | dev_macro_f1=0.5000 | dev_accuracy=0.5333
Epoch 04 | loss=0.8750 | dev_macro_f1=0.5273 | dev_accuracy=0.5607
Epoch 05 | loss=0.6725 | dev_macro_f1=0.5696 | dev_accuracy=0.6087
Epoch 06 | loss=0.5069 | dev_macro_f1=0.5962 | dev_accuracy=0.6372
Epoch 07 | loss=0.3926 | dev_macro_f1=0.5772 | dev_accuracy=0.6186
Epoch 08 | loss=0.3080 | dev_macro_f1=0.5782 | dev_accuracy=0.6142
Early stopping after 8 epochs.
LLM Judge labels are enabled; skipping infobox weak-label override for BiLSTM KG predictions.
Saved BiLSTM outputs to: bilstm


,variant,text_column,dev_accuracy,dev_macro_f1,dev_weighted_f1,test_accuracy,test_macro_f1,test_weighted_f1,train_examples,dev_examples,test_examples,vocab_size,max_len,embedding_dim,hidden_dim,epochs_run,best_dev_macro_f1
0,bilstm_marked,text_marked,0.637158,0.596182,0.63806,0.642775,0.589781,0.644268,4251,915,865,6378,128,100,128,8,0.596182


,epoch,train_loss,dev_accuracy,dev_macro_f1,dev_weighted_f1
3,4,0.874958,0.560656,0.527331,0.565954
4,5,0.672520,0.608743,0.569558,0.615537
5,6,0.506922,0.637158,0.596182,0.638060
6,7,0.392576,0.618579,0.577197,0.626213
7,8,0.307984,0.614208,0.578205,0.617331


,head,tail,label,predicted_label,confidence,context
0,Magdalena,Werdekraf,romantic,family,0.579004,"She also often trained with her brother Werdekraf before she moved to live at the Royal Palace, following her marriage."
1,Lasfam,Ferdinand,no_relation,service_retainer,0.522549,"When Ferdinand personally asked him to do as the others did so Lasfam wouldn't have to suffer so much, Lasfam refused, stating that doin..."
2,Liebeskhilfe,Hannelore,romantic,romantic,0.942973,"Dregarnuhr appears to have some reservations, but Liebeskhilfe pushes Hannelore into Wentuchte's tapestry, before the other goddess can ..."
3,Gundolf,Dregarnuhr,romantic,no_relation,0.985325,"Hannelore reports that she gained the protection of Angriff, the God of War, and Dregarnuhr, the Goddess of Time, albeit she is very con..."
4,Curtiss,Ferdinand,enemy_rival,enemy_rival,0.748227,"Curtiss lures Immanuel out of his office for them, whom is quickly apprehended by bands of light by Ferdinand."
5,Rauchelstra,Schwartz and Weiss,service_retainer,no_relation,0.561643,"To further ensure the safety of this new system, she created Schwartz and Weiss as guardians, but disguised them as library magic tools,..."
6,Viscount Joisontak,Charlotte,romantic,no_relation,0.751322,"When called in for questioning, he claims that his only target was Charlotte and that he never intended for Rozemyne to be harmed."
7,Theodore,Giebe Kirnberger,no_relation,enemy_rival,0.478045,"A little later, Giebe Kirnberger negotiates with Aub Ehrenfest, and this arrangement is put into place in exchange for Kirnberger being ..."
8,Curtiss,Ferdinand,friend_ally,no_relation,0.768979,Curtiss guides Ferdinand to the storage room where the Temple keeps its medals.
9,Ewigeliebe,Flutrane,no_relation,romantic,0.834373,"She found Geduldh trapped and melted the ice with her rays of sunlight, and Flutrane caused the water to flood away, bringing spring to ..."


## 12. Choose the best baseline variant and predict all candidates

The best baseline variant is selected by dev macro-F1. Predictions for all candidate pairs are saved for later KG construction.

In [6]:
best_variant = metrics_df.iloc[0]["variant"]
best_text_column = metrics_df.iloc[0]["text_column"]
best_model = trained_models[best_variant]["model"]

print(f"Best variant by dev macro-F1: {best_variant} using {best_text_column}")

all_pred = candidates_df.copy()
all_pred["predicted_label"] = best_model.predict(all_pred[best_text_column])

if hasattr(best_model.named_steps["clf"], "predict_proba"):
    probs = best_model.predict_proba(all_pred[best_text_column])
    classes = best_model.named_steps["clf"].classes_
    all_pred["confidence"] = probs.max(axis=1)
    for i, label in enumerate(classes):
        all_pred[f"prob_{label}"] = probs[:, i]
else:
    all_pred["confidence"] = np.nan

if LABEL_SOURCE == "weak_labels":
    structured_mask = (
        (all_pred["source_type"] == "infobox")
        & (all_pred["weak_label"] != NO_RELATION_LABEL)
    )

    all_pred.loc[structured_mask, "predicted_label"] = all_pred.loc[structured_mask, "weak_label"]
    all_pred.loc[structured_mask, "confidence"] = 0.95
else:
    print("LLM Judge labels are enabled; skipping infobox weak-label override for baseline KG predictions.")

all_predictions_path = BASELINE_DIR / "predictions_all_best_baseline.csv"
all_pred.to_csv(all_predictions_path, index=False)
print(f"Saved all predictions to: {all_predictions_path}")
display(all_pred[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))

Best variant by dev macro-F1: tfidf_logreg_basic using text_basic
LLM Judge labels are enabled; skipping infobox weak-label override for baseline KG predictions.
Saved all predictions to: baseline\predictions_all_best_baseline.csv


,head,tail,label,predicted_label,confidence,context
0,Georgine,Sylvester,family,family,0.981258,"She is the mother of Detlinde, Sylvester's older sister and a former archduke candidate of Ehrenfest."
1,Magdalena,Werdekraf,romantic,family,0.496768,"She also often trained with her brother Werdekraf before she moved to live at the Royal Palace, following her marriage."
2,Arno,Ferdinand,enemy_rival,enemy_rival,0.381629,"Ferdinand, displeased at Arno's actions, had Arno killed."
3,Magdalena,Hildebrand,no_relation,no_relation,0.765668,The boy is Hildebrand and was raised to become a vassal to whichever of his two older brothers would win the position of successor.
4,Wilfried,Veronica,family,family,0.772191,Upon Veronica's imprisonment Wilfried was told that his grandmother had fallen ill and had been moved to a far away place to recover.
5,Lasfam,Ferdinand,no_relation,service_retainer,0.441878,"When Ferdinand personally asked him to do as the others did so Lasfam wouldn't have to suffer so much, Lasfam refused, stating that doin..."
6,Eckhart,Lamprecht,family,family,0.795527,"Despite despising Veroncia as well, his father still didn't want this to come to pass and when he learned of Eckhart's plans intervened ..."
7,Lungtase,Raufereg,family,family,0.804484,"Her less well behaved brother Raufereg has to stay home, since his parents don't want to risk embarrassing themselves and by extension t..."
8,Gloria,Rozemyne,enemy_rival,friend_ally,0.743019,"When Rozemyne enters noble society, Gloria holds her in contempt and often slanders her reputation at tea parties and social gatherings."
9,Georgine,Veronica,no_relation,no_relation,0.778359,"While Veronica was frozen in terror, one of her attendants had immediately sprung into action and administered an antidote."
